# Charlson Comorbidity Index, Length of Stay, and Mortality

This notebook compares adult hematologic-malignancy inpatient discharges with documented sepsis (`A41*`) against those without documented sepsis. Run all cells to refresh the aggregate results from the cached Phase 1–2 cohort.

## Definitions and statistical methods

- Charlson Comorbidity Index: Quan ICD-10 mapping with original Charlson weights; cancer and metastatic-cancer components excluded; no age points; standard diabetes and liver-disease hierarchy applied.
- Descriptive means, SDs, counts, and percentages use `DISCWT`.
- Continuous comparisons use two-sided Welch t-tests; categorical comparisons use overall Pearson chi-square tests. Tests use sampled discharge records and do not account for hospital clustering.
- Mortality percentages exclude records with missing `DIED`; LOS summaries exclude missing `LOS`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.phase_5_cci_los_mortality import main
summary = main()

{
  "unit": "NIS inpatient discharge, not unique patient",
  "cci_definition": "Quan ICD-10 mapping; original Charlson weights; cancer and metastatic cancer excluded; no age points; hierarchy applied.",
  "inference_note": "Descriptive values use DISCWT. P-values use sampled discharge counts and do not account for hospital clustering.",
  "no_sepsis_unweighted_n": 836571,
  "sepsis_unweighted_n": 158421,
  "los_missing": {
    "no_sepsis": 17,
    "sepsis": 7
  },
  "died_missing": {
    "no_sepsis": 375,
    "sepsis": 66
  },
  "table": [
    {
      "outcome": "Charlson Comorbidity Index excluding cancer",
      "level": "Mean (SD)",
      "no_sepsis": "1.84 (1.90)",
      "sepsis": "2.09 (1.92)",
      "p_value": "<0.001",
      "test": "Welch t-test"
    },
    {
      "outcome": "Charlson category",
      "level": "0",
      "no_sepsis": "1,319,600 (31.55%)",
      "sepsis": "190,300 (24.02%)",
      "p_value": "<0.001",
      "test": "Pearson chi-square"
    },
    {
      "outco

In [2]:
table = pd.read_csv(REPO_ROOT / 'outputs/phase_5/clinical_outcomes_by_sepsis.csv', keep_default_na=False)
table.columns = ['Outcome', 'Level', 'No sepsis', 'Sepsis', 'P-value', 'Test']
display(table.style.hide(axis='index'))

Outcome,Level,No sepsis,Sepsis,P-value,Test
Charlson Comorbidity Index excluding cancer,Mean (SD),1.84 (1.90),2.09 (1.92),<0.001,Welch t-test
Charlson category,0,"1,319,600 (31.55%)","190,300 (24.02%)",<0.001,Pearson chi-square
,1–2,"1,616,779 (38.65%)","327,450 (41.34%)",,
,≥3,"1,246,475 (29.80%)","274,355 (34.64%)",,
"Length of stay, days",Mean (SD),6.88 (8.20),11.13 (13.45),<0.001,Welch t-test
In-hospital mortality,Died,"132,635 (3.17%)","136,050 (17.18%)",<0.001,Pearson chi-square


## Unweighted sample results

This table reports actual sampled NIS discharge records without `DISCWT`. It describes the NIS sample rather than national hospitalization estimates.

In [3]:
unweighted_table = pd.read_csv(REPO_ROOT / 'outputs/phase_5/clinical_outcomes_by_sepsis_unweighted.csv', keep_default_na=False)
unweighted_table.columns = ['Outcome', 'Level', 'No sepsis', 'Sepsis', 'P-value', 'Test']
display(unweighted_table.style.hide(axis='index'))

Outcome,Level,No sepsis,Sepsis,P-value,Test
Charlson Comorbidity Index excluding cancer,Mean (SD),1.84 (1.90),2.09 (1.92),<0.001,Welch t-test
Charlson category,0,"263,920 (31.55%)","38,060 (24.02%)",<0.001,Pearson chi-square
,1–2,"323,356 (38.65%)","65,490 (41.34%)",,
,≥3,"249,295 (29.80%)","54,871 (34.64%)",,
"Length of stay, days",Mean (SD),6.88 (8.20),11.13 (13.45),<0.001,Welch t-test
In-hospital mortality,Died,"26,527 (3.17%)","27,210 (17.18%)",<0.001,Pearson chi-square


## Missing-data review

In [4]:
missing = pd.DataFrame([
    {'Variable': 'Length of stay', 'No sepsis missing': summary['los_missing']['no_sepsis'], 'Sepsis missing': summary['los_missing']['sepsis']},
    {'Variable': 'In-hospital mortality', 'No sepsis missing': summary['died_missing']['no_sepsis'], 'Sepsis missing': summary['died_missing']['sepsis']},
])
display(missing.style.hide(axis='index'))

Variable,No sepsis missing,Sepsis missing
Length of stay,17,7
In-hospital mortality,375,66


## Copy/paste-friendly Markdown

The cell below prints the table as plain Markdown for direct use in a report.

In [5]:
def print_markdown(dataframe, title):
    print(f'## {title}\n')
    headers = list(dataframe.columns)
    print('| ' + ' | '.join(headers) + ' |')
    print('|' + '|'.join(['---'] * len(headers)) + '|')
    for row in dataframe.astype(str).itertuples(index=False, name=None):
        print('| ' + ' | '.join(value.replace('|', '\\|') for value in row) + ' |')
    print()

print_markdown(table, 'Weighted national estimates')
print_markdown(unweighted_table, 'Unweighted sample results')

## Weighted national estimates

| Outcome | Level | No sepsis | Sepsis | P-value | Test |
|---|---|---|---|---|---|
| Charlson Comorbidity Index excluding cancer | Mean (SD) | 1.84 (1.90) | 2.09 (1.92) | <0.001 | Welch t-test |
| Charlson category | 0 | 1,319,600 (31.55%) | 190,300 (24.02%) | <0.001 | Pearson chi-square |
|  | 1–2 | 1,616,779 (38.65%) | 327,450 (41.34%) |  |  |
|  | ≥3 | 1,246,475 (29.80%) | 274,355 (34.64%) |  |  |
| Length of stay, days | Mean (SD) | 6.88 (8.20) | 11.13 (13.45) | <0.001 | Welch t-test |
| In-hospital mortality | Died | 132,635 (3.17%) | 136,050 (17.18%) | <0.001 | Pearson chi-square |

## Unweighted sample results

| Outcome | Level | No sepsis | Sepsis | P-value | Test |
|---|---|---|---|---|---|
| Charlson Comorbidity Index excluding cancer | Mean (SD) | 1.84 (1.90) | 2.09 (1.92) | <0.001 | Welch t-test |
| Charlson category | 0 | 263,920 (31.55%) | 38,060 (24.02%) | <0.001 | Pearson chi-square |
|  | 1–2 | 323,356 (38.65%) | 65,490 (41.34%) |  |  

## Suggested interpretation

HM discharges with documented sepsis had a higher mean cancer-excluded Charlson score, a longer mean hospital stay, and higher in-hospital mortality than HM discharges without documented sepsis. Each comparison had p<0.001. Because these are unadjusted comparisons and the inferential tests do not incorporate hospital clustering, they should be interpreted as descriptive preliminary findings rather than causal or final survey-adjusted estimates.